In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

In [2]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df.shape

(10000, 11)

In [4]:
df.columns.tolist()

['model_year',
 'origin',
 'fuel_type',
 'drivetrain',
 'num_doors',
 'engine_displacement',
 'num_cylinders',
 'horsepower',
 'vehicle_weight',
 'acceleration',
 'fuel_efficiency_mpg']

In [5]:
print(df.isnull().sum())

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64


In [7]:
# Filling missing values with zeros.
df = df.fillna(0)
print(df.isnull().sum())

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64


In [8]:
target = "fuel_efficiency_mpg"
y = df[target]
X = df.drop(columns=[target])

In [9]:

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train[target]
y_val = df_val[target]

In [10]:
dv = DictVectorizer(sparse=True)

train_dicts = df_train.drop(columns=[target]).to_dict(orient="records")
val_dicts = df_val.drop(columns=[target]).to_dict(orient="records")

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])
print("Number of features after DictVectorizer:", X_train.shape[1])

Training rows: 6000
Validation rows: 2000
Number of features after DictVectorizer: 16


## Question 1
Train a decision tree regressor with `max_depth=1.`

Which original feature is used for splitting the data?
- `vehicle_weight`
- `model_year`
- `origin`
- `fuel_type`

In [14]:
print("Split feature from DictVectorizer:", split_feature)
print("Original feature:", original_feature)

Split feature from DictVectorizer: model_year
Original feature: model_year


## Question 2
Train a random forest regressor with:
- `n_estimators=10`
- `random_state=1`
- `n_jobs=-1`
  
What is its `RMSE` on the validation data?

In [15]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1,)
rf.fit(X_train, y_train)

val_predictions = rf.predict(X_val)
rmse_q2 = mean_squared_error(y_val, val_predictions) ** 0.5

print(f"Validation RMSE: {rmse_q2:.4f}")

Validation RMSE: 1.8371


## Question 3
Now let's experiment with the `n_estimators` parameter

Try these values of this parameter: `[10, 50, 100, 150].`

Set random_state to `1`.

Evaluate the model on the validation dataset.

Which value gives the lowest validation RMSE? Consider 3 decimal places for calculating the answer.

In [16]:
estimators_values = [10, 50, 100, 150]
rmse_by_estimators = {}

for n in estimators_values:
    model = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    model.fit(X_train, y_train)
    predictions = model.predict(X_val)
    rmse = mean_squared_error(y_val, predictions) ** 0.5
    rmse_by_estimators[n] = rmse
    
    print(f"n_estimators={n:3d}  validation RMSE={rmse:.4f}")

best_estimators = min(rmse_by_estimators, key=rmse_by_estimators.get)
print("\nBest n_estimators:", best_estimators)

n_estimators= 10  validation RMSE=1.8371
n_estimators= 50  validation RMSE=1.7716
n_estimators=100  validation RMSE=1.7681
n_estimators=150  validation RMSE=1.7688

Best n_estimators: 100


## Question 4
Let's select the best max_depth:

- Try different values of `max_depth`: `[10, 15, 20, 25]`
- For each of these values,
  - try different values of `n_estimators` from `[10, 50, 100, 150]`
  - calculate the mean `RMSE`
- Fix the random seed: `random_state=1`

What's the best max_depth, using the mean RMSE?

In [17]:
depth_values = [10, 15, 20, 25]
estimators_values = [10, 50, 100, 150]
mean_rmse_by_depth = {}

for depth in depth_values:
    depth_rmses = []

    for n in estimators_values:
        model = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        model.fit(X_train, y_train)
        predictions = model.predict(X_val)
        rmse = mean_squared_error(y_val, predictions) ** 0.5
        depth_rmses.append(rmse)

    mean_rmse = float(np.mean(depth_rmses))
    mean_rmse_by_depth[depth] = mean_rmse
    print(f"max_depth={depth:2d}  mean RMSE={mean_rmse:.4f}")

best_depth = min(mean_rmse_by_depth, key=mean_rmse_by_depth.get)
print("\nBest max_depth:", best_depth)

max_depth=10  mean RMSE=1.7554
max_depth=15  mean RMSE=1.7840
max_depth=20  mean RMSE=1.7882
max_depth=25  mean RMSE=1.7863

Best max_depth: 10


## Question 5

We can extract feature importance information from tree-based models.

At each step of the decision tree learning algorithm, it finds the best split. When doing it, we can calculate "gain" - the reduction in impurity before and after the split. This gain is quite useful in understanding what are the important features for tree-based models.

In Scikit-Learn, tree-based models contain this information in the `feature_importances_` field.

For this homework question, we'll find the most important feature:

- Train the model with these parameters:
  - `n_estimators=10,`
  - `max_depth=20,`
  - `random_state=1,`
  - `n_jobs=-1` (optional)
- Get the feature importance information from this model
  
What's the most important feature (among these 4)?

In [18]:
rf_q5 = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_q5.fit(X_train, y_train)

feature_names = dv.get_feature_names_out()
importances = rf_q5.feature_importances_

# Sum importance for one-hot columns that belong to the same original feature.
importance_by_original_feature = {}
for name, importance in zip(feature_names, importances):
    original_name = name.split("=", 1)[0]
    importance_by_original_feature[original_name] = (importance_by_original_feature.get(original_name, 0.0) + importance)

for feature in ["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]:
    print(f"{feature}: {importance_by_original_feature.get(feature, 0.0):.4f}")

best_feature = max(["vehicle_weight", "horsepower", "acceleration", "engine_displacement"],key=lambda feature: importance_by_original_feature.get(feature, 0.0))
print("\nMost important feature:", best_feature)

vehicle_weight: 0.2039
horsepower: 0.0780
acceleration: 0.0552
engine_displacement: 0.0610

Most important feature: vehicle_weight


## Question 6
Now let's train an XGBoost model! For this question, we'll tune the `eta` parameter:

Install XGBoost
- Create DMatrix for train and validation
- Create a watchlist
- Train a model with these parameters for 100 rounds:
  
``` 
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}
```
Now change eta from `0.3` to `0.1`.

Which eta leads to the best RMSE score on the validation dataset?

In [19]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
watchlist = [(dtrain, "train"), (dval, "validation")]

xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1,
}

xgb_results = {}

for eta in [0.3, 0.1]:
    params = xgb_params.copy()
    params["eta"] = eta

    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)

    predictions = model.predict(dval)
    rmse = mean_squared_error(y_val, predictions) ** 0.5
    xgb_results[eta] = rmse
    print(f"eta={eta}: validation RMSE={rmse:.4f}")

if np.isclose(xgb_results[0.3], xgb_results[0.1]):
    print("\nBest eta: Both give equal value")
else:
    print("\nBest eta:", min(xgb_results, key=xgb_results.get))


eta=0.3: validation RMSE=1.8265
eta=0.1: validation RMSE=1.7248

Best eta: 0.1
